# Eco-Logistics v2 — GRPO Training (Colab)

**v9 notebook** — trains GRPO against the v2 environment with:
- 25-step episodes, curriculum learning, receding-horizon replanning
- Multi-component env (single-agent training; multi-agent endpoints stay live)
- Grader-only reward (avoids the SFT-then-GRPO collapse documented in v8 writeup)

**Run cells top-to-bottom.** Stop at any cell that fails — output will tell you what to fix.


In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
# T4 / L4 friendly. ~3-4 min on first run.
%%capture
!pip install -q --no-deps unsloth_zoo==2024.10.0
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade trl==0.24.0 peft transformers accelerate datasets
!pip install -q requests pydantic huggingface_hub

print("✓ install complete")


In [ ]:
# ── Cell 2: Config + imports (v9 FINAL — hard task focused) ─────────────────
import os, json, time, random, re
from typing import Any, Dict, List, Optional, Tuple
import requests

# ─── EDIT THESE ──────────────────────────────────────────────────────────────
ENV_URL          = "https://lokeshrao226-eco-logistics.hf.space"
HF_TOKEN         = ""          # ← paste your token here if you want to push later
HUB_REPO         = "lokeshrao226/eco-logistics-qwen-grpo-v2"
BASE_MODEL       = "Qwen/Qwen2.5-1.5B-Instruct"
PUSH_TO_HUB      = False

# Training hyperparameters
MAX_STEPS               = 40          # enough for clear upward curve
NUM_GENERATIONS         = 4
PER_DEVICE_BATCH        = 1
GRAD_ACCUM_STEPS        = 4
LEARNING_RATE           = 1.5e-6
MAX_PROMPT_LENGTH       = 1024
MAX_COMPLETION_LENGTH   = 1024
N_UNIQUE_PROMPTS        = 80
TRAIN_SEED_RANGE        = (0, 120)

# v2 specific — FORCE hard task
PLAN_HORIZON            = 25
REPLAN_INTERVAL         = 4
USE_CURRICULUM          = False       # Disabled to avoid easy-task overfitting
FORCE_HARD_TASK_PROB    = 0.6         # 60% net_zero_profit, 40% inventory_balanced

# Reward
GRADER_SCALE            = 10000.0
FORMAT_PENALTY_INVALID  = -1000.0

# Output
OUTPUT_DIR              = "/content/grpo-eco-logistics-v2"
FINAL_LORA_DIR          = "/content/grpo-eco-logistics-v2-lora"

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    !huggingface-cli login --token $HF_TOKEN

print(f"✓ v9 FINAL config loaded")
print(f"  MAX_STEPS = {MAX_STEPS} | FORCE_HARD_TASK_PROB = {FORCE_HARD_TASK_PROB} | CURRICULUM = {USE_CURRICULUM}")


In [ ]:
# ── Cell 3: V2 HTTP client + prompt/parse helpers ─────────────────────────
# Talks to the v2 Space endpoints: /reset_v2, /submit_plan, /run_chunk, /grader

class V2EnvClient:
    def __init__(self, base_url: str, session_id: str = "train-v2"):
        self.base_url = base_url.rstrip("/")
        self.session_id = session_id

    @property
    def headers(self):
        return {"x-session-id": self.session_id, "Content-Type": "application/json"}

    def _post(self, path, payload=None, timeout=30):
        for attempt in range(3):
            try:
                r = requests.post(f"{self.base_url}{path}", json=payload, headers=self.headers, timeout=timeout)
                r.raise_for_status()
                return r.json()
            except Exception as e:
                if attempt == 2:
                    raise
                time.sleep(0.5 * (attempt + 1))

    def _get(self, path, timeout=30):
        for attempt in range(3):
            try:
                r = requests.get(f"{self.base_url}{path}", headers=self.headers, timeout=timeout)
                r.raise_for_status()
                return r.json()
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(0.5 * (attempt + 1))

    def reset_v2(self, task_id, seed):       return self._post("/reset_v2", {"task_id": task_id, "seed": seed, "use_v2": True})
    def reset_curriculum(self, seed):         return self._post("/reset_curriculum", {"seed": seed})
    def curriculum_state(self):               return self._get("/curriculum_state")
    def force_curriculum_level(self, level):  return self._post(f"/force_curriculum_level?level={level}")
    def submit_plan(self, plan_dict):         return self._post("/submit_plan", {"plan": plan_dict})
    def submit_revised_plan(self, plan_dict): return self._post("/submit_revised_plan", {"plan": plan_dict})
    def run_chunk(self, chunk_size=None):
        path = "/run_chunk" if chunk_size is None else f"/run_chunk?chunk_size={chunk_size}"
        return self._post(path)
    def grader(self, task_id):                return self._post("/grader", {"task_id": task_id})
    def health(self):                         return self._get("/health")


# ─── Prompt builders (v1-compatible ChatML) ────────────────────────────────
V2_SYSTEM_PROMPT = (
    "You are a supply chain AI managing 3 warehouses. Output ONLY valid JSON array "
    "of plan steps. Each step: ship_amount, origin_city, destination_city, "
    "speed_mode ('Rail' or 'Air'). Always prefer rail. Parse weather_alert defensively."
)

def format_obs_prompt(obs):
    if hasattr(obs, "model_dump"):
        obs = obs.model_dump()
    return (
        f"State at step {obs.get('step_number', 0)}/{obs.get('total_steps', 25)}:\n"
        f"  Inventory: {obs.get('current_inventory', {})}\n"
        f"  Demand:    {obs.get('current_demand', {})}\n"
        f"  Pending:   {obs.get('pending_shipments', [])}\n"
        f"  Carbon balance: {obs.get('carbon_credit_balance', 0)}\n"
        f"  Weather: {obs.get('weather_alert') or 'none'}\n\n"
        f"Output a JSON array of {PLAN_HORIZON} actions for the rest of the episode."
    )

def build_chat_prompt(user_body, system=V2_SYSTEM_PROMPT):
    return (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{user_body}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


# ─── Plan parser ────────────────────────────────────────────────────────────
SAFE_FALLBACK_STEP = {
    "ship_amount": 0.0, "origin_city": "Seattle", "destination_city": "Chicago",
    "speed_mode": "Rail",
}

# ─── City coercer + STRICT parser (forces Rail + caps ship_amount) ───────
def _coerce_city(c, default):
    """Map model output to a valid city. Substring match for fuzzy matches."""
    if not c:
        return default
    c = str(c).strip()
    if c in ["Seattle", "Chicago", "NYC"]:
        return c
    cl = c.lower()
    for canonical in ["Seattle", "Chicago", "NYC"]:
        if canonical.lower() in cl or cl in canonical.lower():
            return canonical
    return default

def parse_plan(completion: str, target_length: int = PLAN_HORIZON) -> Tuple[List[Dict], bool]:
    """CONSTRAINED parser: forces Rail-only and caps ship_amount at 5.
    Prevents carbon overshoot — model was emitting Air shipments that destroy the budget.
    """
    if not completion:
        return [SAFE_FALLBACK_STEP] * target_length, False
    match = re.search(r"\[.*\]", completion, re.DOTALL)
    if not match:
        return [SAFE_FALLBACK_STEP] * target_length, False
    try:
        parsed = json.loads(match.group(0))
        if not isinstance(parsed, list) or len(parsed) == 0:
            return [SAFE_FALLBACK_STEP] * target_length, False
        cleaned = []
        for item in parsed:
            if not isinstance(item, dict):
                continue
            try:
                origin = _coerce_city(item.get("origin_city", ""), "Seattle")
                dest = _coerce_city(item.get("destination_city", ""), "Chicago")
                if origin == dest:
                    dest = "Chicago" if origin != "Chicago" else "NYC"
                cleaned.append({
                    "ship_amount": min(2.0, max(0.0, float(item.get("ship_amount", 0.0)))),
                    "origin_city": origin,
                    "destination_city": dest,
                    "speed_mode": "Rail",   # FORCED — Air kills carbon score
                })
            except Exception:
                continue
        if len(cleaned) == 0:
            return [SAFE_FALLBACK_STEP] * target_length, False
        while len(cleaned) < target_length:
            cleaned.append(SAFE_FALLBACK_STEP.copy())
        return cleaned[:target_length], True
    except (json.JSONDecodeError, ValueError):
        return [SAFE_FALLBACK_STEP] * target_length, False


# ─── Verify Space is reachable ──────────────────────────────────────────────
client = V2EnvClient(ENV_URL)
try:
    h = client.health()
    print(f"✓ Space reachable: {h}")
    cs = client.curriculum_state()
    print(f"✓ curriculum endpoint works: level={cs.get('current_level', '?')}")
    # Confirm /reset_v2 actually exists
    r = client.reset_v2("net_zero_profit", seed=999)
    assert r.get("total_steps") == 25, f"v2 endpoint should give 25 steps, got {r.get('total_steps')}"
    print(f"✓ /reset_v2 returns 25-step task")
except Exception as e:
    print(f"✗ FAIL — Space not reachable or v2 endpoints missing: {e}")
    print(f"  Fix: confirm Space at {ENV_URL} has v2 main.py deployed and is green.")
    raise


In [ ]:
# Cell 0: Fix unsloth/unsloth_zoo version mismatch
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install -U torchao>=0.16.0

In [ ]:
# ── FIX: Install bitsandbytes (required by Unsloth 4-bit) ─────────────────
!pip install --no-deps bitsandbytes>=0.46.1

print("✓ bitsandbytes installed")

In [ ]:
# ── Cell 4: Load Qwen + add LoRA ─────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = 2048,
    load_in_4bit    = True,
    dtype           = None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16, lora_dropout = 0, bias = "none",
)
FastLanguageModel.for_training(model)
print(f"✓ Model loaded: {BASE_MODEL} + LoRA r=16")
print(f"  trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# ── Cell 5: Prompt collection + reward function ─────────────────────────
# Collect N diverse initial-state prompts. Build the GRPO reward function
# (grader-only, the proven non-collapsing config from v8 writeup §4.4).

# ─── Collect prompts ────────────────────────────────────────────────────────
print(f"Collecting {N_UNIQUE_PROMPTS} prompts from v2 env...")
prompts = []
seed_lo, seed_hi = TRAIN_SEED_RANGE
for i, seed in enumerate(range(seed_lo, seed_hi)[:N_UNIQUE_PROMPTS]):
    try:
        if USE_CURRICULUM:
            obs = client.reset_curriculum(seed=seed)
        else:
            obs = client.reset_v2("net_zero_profit", seed=seed)
        body = format_obs_prompt(obs)
        prompts.append({"prompt": build_chat_prompt(body)})
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{N_UNIQUE_PROMPTS} collected")
    except Exception as e:
        print(f"  seed {seed} failed: {e}")
    time.sleep(0.3)

print(f"\n✓ Collected {len(prompts)} prompts")
assert len(prompts) >= 4, "Need >=4 prompts for GRPO. Check Space."

# ─── GRPO reward function ───────────────────────────────────────────────────
# For each completion: parse plan → submit to env → run all chunks → grade.
# Reward = grader_score × 10000  (or -1000 if invalid plan)

global_step_counter = {"n": 0}
training_log = {"step": [], "mean_reward": [], "mean_grader": [], "valid_rate": []}

def grader_only_reward(prompts, completions, **kwargs):
    rewards = []
    valid_count = 0
    grader_scores = []

    for completion in completions:
        plan_list, was_valid = parse_plan(completion, target_length=PLAN_HORIZON)
        if not was_valid:
            rewards.append(FORMAT_PENALTY_INVALID)
            continue

        # Build AgentPlan dict
        plan_dict = {
            "role": "solo",
            "steps": plan_list,
            "is_revision": False,
            "starting_step": 0,
        }

        seed = random.randint(0, 999)
        try:
            # FORCE_HARD_TASK_PROB: 50% chance pick net_zero_profit (the metric that matters)
            if random.random() < FORCE_HARD_TASK_PROB:
                task_id = "net_zero_profit"
            else:
                task_id = "inventory_balanced"
            client.reset_v2(task_id, seed=seed)

            client.submit_plan(plan_dict)

            done = False
            chunks_run = 0
            while not done and chunks_run < 10:  # safety cap
                chunk = client.run_chunk(chunk_size=REPLAN_INTERVAL)
                done = chunk.get("done") or chunk.get("info", {}).get("done", False)
                chunks_run += 1

            g = client.grader(task_id)
            score = g.get("score", 0.0)
            grader_scores.append(score)
            valid_count += 1
            rewards.append(score * GRADER_SCALE)
        except Exception as e:
            print(f"  rollout error: {e}")
            rewards.append(0.0)

    # ─── log batch aggregates ──
    global_step_counter["n"] += 1
    n = len(rewards)
    mean_reward = sum(rewards) / n
    mean_grader = (sum(grader_scores) / len(grader_scores)) if grader_scores else 0.0
    valid_rate = valid_count / n

    training_log["step"].append(global_step_counter["n"])
    training_log["mean_reward"].append(mean_reward)
    training_log["mean_grader"].append(mean_grader)
    training_log["valid_rate"].append(valid_rate)

    if global_step_counter["n"] % 5 == 0:
        print(f"[step {global_step_counter['n']}] reward={mean_reward:.0f}  grader={mean_grader:.3f}  valid={valid_rate:.0%}")

    return rewards

print("\n✓ Reward function defined")


In [ ]:
# ── Cell 6: GRPO training loop + save LoRA ───────────────────────────────
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

train_dataset = Dataset.from_list(prompts)

grpo_config = GRPOConfig(
    output_dir                  = OUTPUT_DIR,
    max_prompt_length           = MAX_PROMPT_LENGTH,
    max_completion_length       = MAX_COMPLETION_LENGTH,
    per_device_train_batch_size = PER_DEVICE_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    num_generations             = NUM_GENERATIONS,
    learning_rate               = LEARNING_RATE,
    max_steps                   = MAX_STEPS,
    logging_steps               = 5,
    save_steps                  = 10,
    report_to                   = "none",
    seed                        = 42,
    remove_unused_columns       = False,
)

trainer = GRPOTrainer(
    model         = model,
    args          = grpo_config,
    reward_funcs  = [grader_only_reward],
    train_dataset = train_dataset,
    tokenizer     = tokenizer,
)

print("=" * 70)
print(f"Starting GRPO v2 training")
print(f"  steps          : {MAX_STEPS}")
print(f"  effective batch: {PER_DEVICE_BATCH * GRAD_ACCUM_STEPS}")
print(f"  generations    : {NUM_GENERATIONS}/prompt")
print(f"  curriculum     : {USE_CURRICULUM}")
print(f"  plan horizon   : {PLAN_HORIZON} steps")
print("=" * 70)

trainer.train()

print("\n" + "=" * 70)
print("Training complete.")
print("=" * 70)

# Save LoRA
model.save_pretrained(FINAL_LORA_DIR)
tokenizer.save_pretrained(FINAL_LORA_DIR)
print(f"✓ saved to {FINAL_LORA_DIR}")

# Save training log
with open("/content/training_log_v2.json", "w") as f:
    json.dump(training_log, f, indent=2)
print(f"✓ training log saved to /content/training_log_v2.json")

# Final curriculum state
try:
    cs = client.curriculum_state()
    print(f"\nFinal curriculum state: {cs}")
except Exception:
    pass

# Optional: push to Hub
if PUSH_TO_HUB and HUB_REPO:
    print(f"\nPushing LoRA to {HUB_REPO}...")
    try:
        model.push_to_hub(HUB_REPO, token=os.environ.get("HF_TOKEN"))
        tokenizer.push_to_hub(HUB_REPO, token=os.environ.get("HF_TOKEN"))
        print(f"✓ pushed to https://huggingface.co/{HUB_REPO}")
    except Exception as e:
        print(f"  push failed: {e}")


In [ ]:
# ── Cell 6.5: Backup LoRA + logs to Google Drive (run RIGHT AFTER training) ─
# So you don't lose work if session dies. ~1 min.
from google.colab import drive
drive.mount('/content/drive')
import shutil, os

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/eco-logistics-v9-backup"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

if os.path.exists(FINAL_LORA_DIR):
    shutil.copytree(FINAL_LORA_DIR, f"{DRIVE_BACKUP_DIR}/v9-lora", dirs_exist_ok=True)
    print(f"✓ LoRA backed up to {DRIVE_BACKUP_DIR}/v9-lora")

if os.path.exists("/content/training_log_v2.json"):
    shutil.copy("/content/training_log_v2.json", f"{DRIVE_BACKUP_DIR}/training_log_v2.json")
    print(f"✓ training log backed up")

print("\n✓ All artifacts saved to Drive — safe from session timeout.")


In [ ]:
# ── Cell 7: Held-out eval on ALL 3 TASKS × 10 SEEDS each ────────────────
FastLanguageModel.for_inference(model)

def eager_generate(prompt, temperature=0.0, max_new_tokens=MAX_COMPLETION_LENGTH):
    """Greedy decoding for deterministic eval — measure THE policy, not random samples."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=False,   # GREEDY
    )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def run_one_held_out_episode(seed, task_id):
    """Run one fresh episode on a fixed task. Returns dict with grader, valid, etc."""
    obs = client.reset_v2(task_id, seed=seed)
    body = format_obs_prompt(obs)
    prompt = build_chat_prompt(body)
    completion = eager_generate(prompt, temperature=0.3)
    plan_list, valid = parse_plan(completion, target_length=PLAN_HORIZON)

    plan_dict = {"role": "solo", "steps": plan_list, "is_revision": False, "starting_step": 0}
    try:
        client.submit_plan(plan_dict)
    except Exception as e:
        return {"seed": seed, "task": task_id, "valid": valid, "grader": 0.0, "error": str(e)}

    done = False
    chunks = 0
    while not done and chunks < 10:
        chunk = client.run_chunk(chunk_size=REPLAN_INTERVAL)
        done = chunk.get("done", False)
        chunks += 1

    g = client.grader(task_id)
    return {"seed": seed, "task": task_id, "valid": valid, "grader": g["score"]}


print("=" * 70)
print("Held-out eval: 3 tasks × 10 seeds = 30 episodes")
print("=" * 70)

TASKS_TO_EVAL = ["restock_only", "inventory_balanced", "net_zero_profit"]
EVAL_SEEDS = list(range(600, 610))  # seeds 600-609 = 10 seeds

all_results = {t: [] for t in TASKS_TO_EVAL}

for task_id in TASKS_TO_EVAL:
    print(f"\n--- task: {task_id} ---")
    for seed in EVAL_SEEDS:
        try:
            r = run_one_held_out_episode(seed, task_id)
            all_results[task_id].append(r)
            print(f"  seed={seed}  valid={r['valid']}  grader={r['grader']:.3f}")
        except Exception as e:
            print(f"  seed={seed}  ERROR: {e}")
            all_results[task_id].append({"seed": seed, "task": task_id, "valid": False, "grader": 0.0})

# ─── Per-task aggregates ───────────────────────────────────────────────────
print("\n" + "=" * 70)
print("PER-TASK AGGREGATES (held-out)")
print("=" * 70)
print(f"{'Task':<25} {'Mean Grader':<15} {'Valid Rate':<15} {'N':<5}")
print("-" * 70)
for task_id in TASKS_TO_EVAL:
    rs = all_results[task_id]
    n = len(rs)
    mean_g = sum(r["grader"] for r in rs) / max(1, n)
    valid_r = sum(1.0 if r["valid"] else 0.0 for r in rs) / max(1, n)
    print(f"{task_id:<25} {mean_g:<15.3f} {valid_r:<15.0%} {n:<5}")

# ─── Overall ──────────────────────────────────────────────────────────────
all_flat = [r for task_rs in all_results.values() for r in task_rs]
n_total = len(all_flat)
mean_grader_all = sum(r["grader"] for r in all_flat) / n_total
valid_rate_all = sum(1.0 if r["valid"] else 0.0 for r in all_flat) / n_total
print("-" * 70)
print(f"{'OVERALL':<25} {mean_grader_all:<15.3f} {valid_rate_all:<15.0%} {n_total:<5}")
print("=" * 70)

# ─── Save ─────────────────────────────────────────────────────────────────
with open("/content/eval_v2_results_3tasks.json", "w") as f:
    json.dump({
        "per_task": all_results,
        "summary": {
            "tasks": TASKS_TO_EVAL,
            "seeds": EVAL_SEEDS,
            "mean_grader_overall": mean_grader_all,
            "valid_rate_overall": valid_rate_all,
            "per_task_grader": {t: sum(r["grader"] for r in all_results[t]) / max(1, len(all_results[t])) for t in TASKS_TO_EVAL},
            "per_task_valid_rate": {t: sum(1.0 if r["valid"] else 0.0 for r in all_results[t]) / max(1, len(all_results[t])) for t in TASKS_TO_EVAL},
        }
    }, f, indent=2)
print(f"\n✓ saved to /content/eval_v2_results_3tasks.json")